# Notebook 18 operator — learned global summaries and exact local attention

Purpose: define a trainable replacement for the context and EMA encoders inside the complete JEPA system. The predictor, target masking, loss and EMA update remain the existing JEPA components. This is a new interaction rule, not a reproduction of frozen ViT attention.

Inputs: ViT mixer reference and explicit configuration; no images, labels or training are read here. Outputs: tagged reusable operator definitions. Expected runtime: under 10 seconds for definitions. This notebook launches no training and creates no checkpoints. Long-run checkpoints belong to `checkpoints/<run_id>/` and are managed by Notebook 18's runner. Resume definitions by running this notebook top-to-bottom; resume experiments through the Notebook 18 front door and its immutable manifest.

Only cells tagged `ahr-export` may be loaded by another notebook.

In [ ]:
from pathlib import Path
import json,os
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
_boot={'__file__':str(ROOT/'fmm_jepa/bootstrap.ipynb')}
for _cell in json.loads((ROOT/'fmm_jepa/bootstrap.ipynb').read_text())['cells']:
    if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):
        exec(compile(''.join(_cell['source']),'bootstrap#ahr-export','exec'),_boot)
_boot['install_notebook_package'](ROOT)


For each head, learned row directions and query/key directions are normalized. With feature temperature $\beta\in[0.25,4]$,
$$\phi^q_i=\operatorname{softmax}_r(\beta\sqrt d\,\langle \widehat q_i,\widehat w^q_r\rangle),\qquad
\phi^k_j=\operatorname{softmax}_r(\beta\sqrt d\,\langle \widehat k_j,\widehat w^k_r\rangle).$$
The strictly positive global kernel is normalized using exact summaries $S=\sum_j\phi^k_jv_j^T$ and $z=\sum_j\phi^k_j$. Separately, local attention uses raw $q,k$ and exact softmax over present tokens in the original 3×3 patch neighborhood. A learned sigmoid gate gives
$$y_i=(1-g_h)\frac{(\phi_i^q)^TS}{(\phi_i^q)^Tz}+g_h\sum_{j\in\mathcal N(i)}\frac{e^{q_i^Tk_j/\tau_h}}{\sum_{l\in\mathcal N(i)}e^{q_i^Tk_l/\tau_h}}v_j.$$
No far-pair matrix, random exponential approximation, adaptive pivoting or subtraction of approximate local entries is used. Local and global branches may both include local sources; their overlap is explicitly part of the convex-mixture model. Raw Q/K magnitudes influence the local branch. The dense reference is used only for independent software verification.

In [ ]:
"""Learned simplex global summaries plus separately normalized exact local attention.

A new trainable operator, exact for its own explicitly defined convex mixture.
It does not approximate frozen ViT softmax and has no classical FMM certificate.
"""
import copy,math,hashlib
import torch
from torch import nn
from torch.nn import functional as F

BACKEND='learned-simplex-global-exact-local-convex-mixture-v1'
ALLOWED_RANKS=(8,16,32,64,128)
OPTIONS={'rank','feature_seed','local_radius','spatial_mode','feature_temperature_min',
         'feature_temperature_max','feature_temperature_initial','gate_initial'}

def seeded_feature_directions(heads,head_dim,rank,seed,dtype=torch.float32):
    """Independent local CPU generator; no project/global RNG is consumed."""
    if min(int(heads),int(head_dim),int(rank))<1:raise ValueError('Positive dimensions required')
    generator=torch.Generator(device='cpu').manual_seed(int(seed))
    value=torch.randn(2,int(heads),int(rank),int(head_dim),generator=generator,dtype=torch.float64)
    value=value/value.norm(dim=-1,keepdim=True)
    return value[0].to(dtype=dtype),value[1].to(dtype=dtype)

def feature_sha256(value):
    value=value.detach().cpu().contiguous()
    return hashlib.sha256((str(value.dtype)+':'+str(tuple(value.shape))).encode()+value.numpy().tobytes()).hexdigest()

def local_neighbor_indices(indices,valid,grid,local_radius=1,check_inputs=True):
    """One vectorized original-patch lookup; valid duplicates are rejected in checked API.

    Model encoders validate original IDs before block application. Their forward
    uses check_inputs=False to avoid device synchronization at every block.
    """
    if indices.ndim!=2 or valid.shape!=indices.shape or indices.dtype!=torch.long or valid.dtype!=torch.bool:
        raise ValueError('Expected long indices[B,N] and bool validity[B,N]')
    if indices.device!=valid.device:raise ValueError('Position/validity devices differ')
    height,width=map(int,grid);B,N=indices.shape;radius=int(local_radius)
    if height<1 or width<1 or radius not in(0,1):raise ValueError('Positive grid and local radius0or1 required')
    safe=indices.clamp(0,height*width-1)
    if check_inputs:
        if bool((((indices<0)|(indices>=height*width))&valid).any()):raise ValueError('Valid original index outside grid')
        counts=indices.new_zeros((B,height*width));counts.scatter_add_(1,safe,valid.long())
        if bool((counts>1).any()):raise ValueError('Duplicate valid original index')
    # scatter_reduce amax prevents invalid -1 slots from overwriting real slots.
    lookup=indices.new_full((B,height*width),-1)
    slots=torch.arange(N,device=indices.device)[None].expand(B,-1)
    lookup.scatter_reduce_(1,safe,torch.where(valid,slots,-torch.ones_like(slots)),reduce='amax',include_self=True)
    offsets=torch.arange(-radius,radius+1,device=indices.device)
    dr,dc=torch.meshgrid(offsets,offsets,indexing='ij');dr=dr.flatten();dc=dc.flatten()
    row=safe//width;col=safe%width
    rr=row[...,None]+dr;cc=col[...,None]+dc
    inside=(rr>=0)&(rr<height)&(cc>=0)&(cc<width)&valid[...,None]
    original=rr.clamp(0,height-1)*width+cc.clamp(0,width-1)
    neighbors=lookup.gather(1,original.flatten(1)).reshape_as(original)
    near_valid=inside&(neighbors>=0)
    return neighbors.clamp_min(0),near_valid

def _checked_inputs(q,k,v,valid,query_features,key_features,temperature,gate,tau,check_inputs):
    if q.ndim!=4 or k.shape!=q.shape or v.shape[:3]!=q.shape[:3]:raise ValueError('q/k[B,H,N,D], v[B,H,N,Dv] required')
    B,H,N,D=q.shape
    if valid.shape!=(B,N)or valid.dtype!=torch.bool or valid.device!=q.device:raise ValueError('Validity[B,N] on input device required')
    if k.device!=q.device or v.device!=q.device or k.dtype!=q.dtype or v.dtype!=q.dtype:raise ValueError('Input dtype/devices differ')
    if q.dtype not in(torch.float32,torch.float64):raise ValueError('Validated operator arithmetic is FP32 or FP64')
    if query_features is None or key_features is None or query_features.ndim!=3 or key_features.shape!=query_features.shape:
        raise ValueError('Learned query/key directions[H,R,D] required')
    if query_features.shape[0]!=H or query_features.shape[2]!=D:raise ValueError('Feature dimensions mismatch')
    Wq=query_features.to(device=q.device,dtype=q.dtype);Wk=key_features.to(device=q.device,dtype=q.dtype)
    beta=torch.as_tensor(temperature,device=q.device,dtype=q.dtype).reshape(H)
    mix=torch.as_tensor(gate,device=q.device,dtype=q.dtype).reshape(H)
    tau=torch.full((H,),math.sqrt(D),device=q.device,dtype=q.dtype)if tau is None else torch.as_tensor(tau,device=q.device,dtype=q.dtype).reshape(H)
    mask=valid[:,None,:,None]
    Q,K,V=[torch.where(mask,t,torch.zeros_like(t))for t in(q,k,v)]
    if check_inputs:
        if not all(bool(torch.isfinite(t).all())for t in(Q,K,V,Wq,Wk,beta,mix,tau)):raise FloatingPointError('Finite valid inputs/parameters required')
        if bool(((beta<=0)|(beta>4.)).any())or bool(((mix<0)|(mix>1)).any())or bool((tau<=0).any()):raise ValueError('Feature temperature in(0,4], gate in[0,1], tau>0 required')
    return Q,K,V,Wq,Wk,beta,mix,tau

def learned_features(q,k,Wq,Wk,beta,valid):
    """Strictly positive simplex features; bounded logits +/-4sqrt(D).

    Unit Q/K is only used inside the global feature map. Raw projections retain
    their magnitudes in the local branch and in kernel_state diagnostics.
    """
    Q=F.normalize(q,dim=-1,eps=1e-6);K=F.normalize(k,dim=-1,eps=1e-6)
    Wq=F.normalize(Wq,dim=-1,eps=1e-6);Wk=F.normalize(Wk,dim=-1,eps=1e-6)
    logitsq=torch.einsum('bhnd,hrd->bhnr',Q,Wq)*beta[None,:,None,None]*math.sqrt(q.shape[-1])
    logitsk=torch.einsum('bhnd,hrd->bhnr',K,Wk)*beta[None,:,None,None]*math.sqrt(q.shape[-1])
    mask=valid[:,None,:,None]
    return torch.where(mask,logitsq.softmax(-1),torch.zeros_like(logitsq)),torch.where(mask,logitsk.softmax(-1),torch.zeros_like(logitsk))

def _gather_local_kv(k,v,neighbors):
    """Gather concatenated keys/values once; no per-neighbor Python loop."""
    B,H,N,D=k.shape;L=neighbors.shape[-1];joined=torch.cat((k,v),dim=-1)
    gather=neighbors.flatten(1)[:,None,:,None].expand(B,H,N*L,joined.shape[-1])
    gathered=joined.gather(2,gather).reshape(B,H,N,L,joined.shape[-1])
    return gathered[...,:D],gathered[...,D:]

def learned_kernel_attention(q,k,v,valid,positions=None,query_features=None,key_features=None,
                             feature_temperature=None,gate=None,tau=None,indices=None,grid=None,
                             local_radius=1,return_stats=False,check_inputs=True):
    """Exact application of the declared learned kernel mixture in linear memory.

    G_ij=phi_q(i)^Tphi_k(j)/sum_j phi_q(i)^Tphi_k(j).
    L_ij=1[j in original-grid neighborhood(i)] exp(rawq_i rawk_j/tau)
         /sum_local exp(rawq_i rawk_j/tau).
    A_ij=(1-sigmoid(gate_raw_h))*G_ij+sigmoid(gate_raw_h)*L_ij.
    The API gate argument is the actual probability. Both branches are positive,
    row normalized separately, and mixed without approximate-kernel subtraction.
    """
    Q,K,V,Wq,Wk,beta,mix,tau=_checked_inputs(q,k,v,valid,query_features,key_features,feature_temperature,gate,tau,check_inputs)
    B,H,N,D=Q.shape;rank=Wq.shape[1]
    if indices is None or grid is None:raise ValueError('Original indices/grid required')
    neighbors,nearvalid=local_neighbor_indices(indices,valid,grid,local_radius,check_inputs)
    phiq,phik=learned_features(Q,K,Wq,Wk,beta,valid)
    summary=phik.transpose(-1,-2)@V;mass=phik.sum(-2)
    numerator=phiq@summary;denominator=(phiq*mass[:,:,None]).sum(-1)
    # Only empty query rows receive the value1. Never alter a valid denominator.
    denominator=torch.where(valid[:,None],denominator,torch.ones_like(denominator))
    global_output=numerator/denominator[...,None]
    local_keys,local_values=_gather_local_kv(K,V,neighbors)
    logits=torch.einsum('bhnd,bhnld->bhnl',Q,local_keys)/tau[None,:,None,None]
    logits=logits.masked_fill(~nearvalid[:,None],-torch.inf)
    logits=torch.where(valid[:,None,:,None],logits,torch.zeros_like(logits))
    local_weights=logits.softmax(-1)*nearvalid[:,None]
    local_output=torch.einsum('bhnl,bhnld->bhnd',local_weights,local_values)
    output=(1-mix[None,:,None,None])*global_output+mix[None,:,None,None]*local_output
    output=torch.where(valid[:,None,:,None],output,torch.zeros_like(output))
    if check_inputs and not bool(torch.isfinite(output).all()):raise FloatingPointError('Nonfinite output; no numerical validity claim')
    if not return_stats:return output
    stats={'algorithm':BACKEND,'rank':int(rank),'head_dimension':int(D),'local_radius':int(local_radius),
           'local_pairs':int(nearvalid.sum()),'feature_elements':2*B*H*N*rank,
           'global_summary_elements':B*H*rank*(V.shape[-1]+1),
           'dense_reference_entries':B*H*N*N,'far_pair_matrix_constructed':False,
           'adaptive_pivots':False,'gather_count':1,'exact_for_own_kernel':True,'exact_vit_attention':False,
           'kernel_scope':'Convex mixture of normalized learned positive feature kernel and exact normalized original-grid local raw-QK attention',
           'logit_bound':4*math.sqrt(D),'numerical_denominator_clamp':False}
    return output,stats

def dense_learned_kernel_reference(q,k,v,valid,positions=None,query_features=None,key_features=None,
                                  feature_temperature=None,gate=None,tau=None,indices=None,grid=None,
                                  local_radius=1,return_weights=False):
    """Independent N*N mathematical reference only; never called by model forward.

    Independently defines all pair weights and 2-D neighborhoods. It does not call
    learned_features, fast gather/topology or summary-contraction code.
    """
    Q,K,V,Wq,Wk,beta,mix,tau=_checked_inputs(q,k,v,valid,query_features,key_features,feature_temperature,gate,tau,True)
    B,H,N,D=Q.shape
    def unit(t):return t/t.norm(dim=-1,keepdim=True).clamp_min(1e-6)
    phiq=(unit(Q)@unit(Wq).transpose(-1,-2)*beta[None,:,None,None]*math.sqrt(D)).softmax(-1)
    phik=(unit(K)@unit(Wk).transpose(-1,-2)*beta[None,:,None,None]*math.sqrt(D)).softmax(-1)
    pairvalid=valid[:,None,:,None]&valid[:,None,None,:]
    global_weights=(phiq@phik.transpose(-1,-2))*pairvalid
    global_mass=global_weights.sum(-1,keepdim=True)
    global_weights=global_weights/torch.where(valid[:,None,:,None],global_mass,torch.ones_like(global_mass))
    height,width=map(int,grid);row=indices.clamp(0,height*width-1)//width;col=indices.clamp(0,height*width-1)%width
    local=((row[:,:,None]-row[:,None,:]).abs()<=int(local_radius))&((col[:,:,None]-col[:,None,:]).abs()<=int(local_radius))&valid[:,:,None]&valid[:,None,:]
    logits=(Q@K.transpose(-1,-2))/tau[None,:,None,None]
    logits=logits.masked_fill(~local[:,None],-torch.inf)
    logits=torch.where(valid[:,None,:,None],logits,torch.zeros_like(logits))
    local_weights=logits.softmax(-1)*local[:,None]
    weights=(1-mix[None,:,None,None])*global_weights+mix[None,:,None,None]*local_weights
    output=(weights@V)*valid[:,None,:,None]
    return(output,weights)if return_weights else output

def exact_softmax_attention(q,k,v,valid,positions=None,tau=None,**unused):
    """Ordinary optimized ViT reference, distinct from the new learned kernel."""
    H,D=q.shape[1],q.shape[-1]
    tau=torch.full((H,),math.sqrt(D),device=q.device,dtype=q.dtype)if tau is None else torch.as_tensor(tau,device=q.device,dtype=q.dtype)
    mask=valid[:,None,:,None];Q,K,V=[torch.where(mask,t,torch.zeros_like(t))for t in(q,k,v)]
    result=F.scaled_dot_product_attention(Q*(math.sqrt(D)/tau)[None,:,None,None],K,V,attn_mask=valid[:,None,None,:],dropout_p=0.)
    return torch.where(mask,result,torch.zeros_like(result))

class LearnedKernelMixer(nn.Module):
    """Original ViT child/state names remain intact; learned feature/gate additions.

    Initialization uses only a private CPU generator and leaves global RNG intact.
    Feature parameters are registered trainable parameters, copied into the EMA
    teacher and updated by the existing JEPA.update_teacher mechanism.
    """
    def __init__(self,reference,options=None):
        super().__init__();options=dict(options or{})
        if set(options)-OPTIONS:raise ValueError('Unknown learned-kernel options:'+str(sorted(set(options)-OPTIONS)))
        self.dim=int(reference.dim);self.heads=int(reference.heads)
        self.rank=int(options.get('rank',64));self.feature_seed=int(options.get('feature_seed',180064));self.local_radius=int(options.get('local_radius',1))
        if self.rank not in ALLOWED_RANKS or self.local_radius not in(0,1):raise ValueError('Declared feature rank and local radius0/1 required')
        if options.get('spatial_mode','disabled')!='disabled':raise ValueError('Signed spatial bias is disabled for this hypothesis')
        self.feature_temperature_min=float(options.get('feature_temperature_min',.25));self.feature_temperature_max=float(options.get('feature_temperature_max',4.))
        initial=float(options.get('feature_temperature_initial',1.));gate=float(options.get('gate_initial',.5))
        if not all(math.isfinite(t)for t in(self.feature_temperature_min,self.feature_temperature_max,initial,gate)):raise ValueError('Finite feature options required')
        if not 0<self.feature_temperature_min<initial<self.feature_temperature_max<=4:raise ValueError('0<min<initial<max<=4 required')
        if not 0<gate<1:raise ValueError('Initial local gate must be strictly inside(0,1)')
        for name,module in reference.named_children():self.add_module(name,copy.deepcopy(module))
        for name,param in reference._parameters.items():
            if param is not None:self.register_parameter(name,nn.Parameter(param.detach().clone(),requires_grad=param.requires_grad))
        for name,value in reference._buffers.items():
            if value is not None:self.register_buffer(name,value.detach().clone())
        device,dtype=self.qkv.weight.device,self.qkv.weight.dtype
        if not hasattr(self,'log_temperature_multiplier'):self.log_temperature_multiplier=nn.Parameter(torch.zeros(self.heads,device=device,dtype=dtype),requires_grad=False)
        if not hasattr(self,'spatial_coefficient_raw'):self.spatial_coefficient_raw=nn.Parameter(torch.zeros(self.heads,device=device,dtype=dtype),requires_grad=False)
        if bool((self.spatial_coefficient_raw.detach()!=0).any()):raise ValueError('Cannot discard incoming nonzero spatial bias')
        self.spatial_coefficient_raw.requires_grad_(False)
        if not hasattr(self,'log_spatial_length'):self.log_spatial_length=nn.Parameter(torch.full((self.heads,),math.log(.35),device=device,dtype=dtype),requires_grad=False)
        self.log_spatial_length.requires_grad_(False)
        self.spatial_coefficient_bound=getattr(reference,'spatial_coefficient_bound',2.)
        Wq,Wk=seeded_feature_directions(self.heads,self.dim//self.heads,self.rank,self.feature_seed,dtype)
        self.query_features=nn.Parameter(Wq.to(device));self.key_features=nn.Parameter(Wk.to(device))
        p=(initial-self.feature_temperature_min)/(self.feature_temperature_max-self.feature_temperature_min)
        self.feature_temperature_raw=nn.Parameter(torch.full((self.heads,),math.log(p/(1-p)),device=device,dtype=dtype))
        self.local_gate_raw=nn.Parameter(torch.full((self.heads,),math.log(gate/(1-gate)),device=device,dtype=dtype))
        self.backend=BACKEND;self.last_operator_stats={}
    def feature_parameters(self,dtype=None):
        dtype=dtype or self.qkv.weight.dtype
        beta=self.feature_temperature_min+(self.feature_temperature_max-self.feature_temperature_min)*self.feature_temperature_raw.to(dtype).sigmoid()
        return {'feature_temperature':beta,'local_gate':self.local_gate_raw.to(dtype).sigmoid()}
    def geometry_parameters(self,dtype=None):
        dtype=dtype or self.qkv.weight.dtype
        multiplier=self.log_temperature_multiplier.to(dtype).clamp(-math.log(16.),math.log(16.)).exp()
        return {'temperature':math.sqrt(self.dim//self.heads)*multiplier,'temperature_multiplier':multiplier,
                'spatial_coefficient':torch.zeros_like(multiplier),'spatial_length':self.log_spatial_length.to(dtype).exp(),**self.feature_parameters(dtype)}
    def _state(self,x,indices,valid,grid):
        if valid is None:valid=torch.ones(x.shape[:2],device=x.device,dtype=torch.bool)
        if indices is None or grid is None:raise ValueError('Original patch indices/grid required')
        safe=torch.where(valid[...,None],x,torch.zeros_like(x));B,N,D=x.shape
        q,k,v=self.qkv(safe).reshape(B,N,3,self.heads,D//self.heads).permute(2,0,3,1,4).unbind(0)
        return q,k,v,valid,self.geometry_parameters(q.dtype)
    def forward(self,x,indices=None,valid=None,grid=None):
        q,k,v,valid,geo=self._state(x,indices,valid,grid)
        output=learned_kernel_attention(q,k,v,valid,query_features=self.query_features,key_features=self.key_features,
            feature_temperature=geo['feature_temperature'],gate=geo['local_gate'],tau=geo['temperature'],
            indices=indices,grid=grid,local_radius=self.local_radius,check_inputs=False)
        return self.projection(output.transpose(1,2).reshape_as(x))*valid[...,None]
    def kernel_state(self,x,indices,valid,grid):
        q,k,v,valid,geo=self._state(x,indices,valid,grid)
        return {'q':q,'k':k,'v':v,'query':q,'key':k,'value':v,'indices':indices,'valid':valid,'grid':grid,
                'query_features':self.query_features,'key_features':self.key_features,'rank':self.rank,**geo,
                'geometry':'New learned positive simplex global kernel plus exact normalized local raw-QK attention',
                'diagnostic_scope':'Raw Q/K/V magnitudes retained; feature normalization only inside global branch; no dense forward matrix'}

def replace_encoder_mixers(model,options=None):
    options=dict(options or{});seed=int(options.get('feature_seed',180064))
    if len(model.context.blocks)!=len(model.target.blocks):raise ValueError('Online and EMA depths differ')
    for layer,(context,target)in enumerate(zip(model.context.blocks,model.target.blocks)):
        current={**options,'feature_seed':seed+layer}
        context.mixer=LearnedKernelMixer(context.mixer,current);target.mixer=LearnedKernelMixer(target.mixer,current)
        target.mixer.load_state_dict(context.mixer.state_dict())
    model.target.requires_grad_(False).eval();return model
